In [ ]:
# %pip install numpy matplotlib torch gymnasium[classic-control] networkx

import random
import time
from collections import deque, namedtuple

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
import gymnasium as gym
import networkx as nx

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch device:", DEVICE)

PyTorch device: cpu


In [ ]:
class RouletteEnv:
    RED_NUMS = {
        1, 3, 5, 7, 9, 12, 14, 16, 18,
        19, 21, 23, 25, 27, 30, 32, 34, 36
    }
    N_ACTIONS = 12

    def __init__(self, seed=SEED):
        self.rng = random.Random(seed)

    def step(self, action):
        spin = self.rng.randint(0, 36)
        if action == 0:                 # Skip
            reward = 0.0
        elif action == 1:               # Red
            reward = 1.0 if spin in self.RED_NUMS else -1.0
        elif action == 2:               # Black
            reward = 1.0 if spin != 0 and spin not in self.RED_NUMS else -1.0
        elif action == 3:               # Odd
            reward = 1.0 if spin != 0 and spin % 2 == 1 else -1.0
        elif action == 4:               # Even
            reward = 1.0 if spin != 0 and spin % 2 == 0 else -1.0
        elif 5 <= action <= 11:         # Single numbers 0 through 6
            reward = 35.0 if spin == action - 5 else -1.0
        else:
            raise ValueError("Action must be in the range 0..11")
        return spin, reward
def train_roulette_qlearning(episodes=20000, alpha=0.05,
                             epsilon=1.0, epsilon_min=0.01, seed=SEED):
    env = RouletteEnv(seed)
    q_values = np.zeros(RouletteEnv.N_ACTIONS, dtype=float)
    rewards = []
    epsilon_decay = (epsilon - epsilon_min) / episodes

    for _ in range(episodes):
        if random.random() < epsilon:
            action = random.randrange(RouletteEnv.N_ACTIONS)
        else:
            action = int(np.argmax(q_values))

        _, reward = env.step(action)
        # One-step action-value update (no future state in this bandit task)
        q_values[action] += alpha * (reward - q_values[action])
        rewards.append(reward)
        epsilon = max(epsilon_min, epsilon - epsilon_decay)

    return q_values, np.asarray(rewards)

roulette_q, roulette_rl_rewards = train_roulette_qlearning()
print("Q-learning action values:", np.round(roulette_q, 3))
print("Best estimated action:", int(np.argmax(roulette_q)))
class RouletteDQN(nn.Module):
    def __init__(self, input_dim=1, output_dim=12):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64), nn.ReLU(),
            nn.Linear(64, 64), nn.ReLU(),
            nn.Linear(64, output_dim)
        )

    def forward(self, x):
        return self.net(x)


def train_roulette_dqn(episodes=12000, batch_size=128, gamma=0.0,
                       lr=1e-3, epsilon_start=1.0, epsilon_end=0.02,
                       epsilon_decay=5000, target_sync=200, seed=SEED):
    # The roulette problem is stateless and each spin is independent.
    # gamma=0 is therefore appropriate: there is no future-state reward.
    env = RouletteEnv(seed)
    policy = RouletteDQN().to(DEVICE)
    target = RouletteDQN().to(DEVICE)
    target.load_state_dict(policy.state_dict())
    optimizer = optim.Adam(policy.parameters(), lr=lr)
    replay = deque(maxlen=10000)
    rewards = []

    for step in range(episodes):
        eps = epsilon_end + (epsilon_start - epsilon_end) * np.exp(-step / epsilon_decay)
        state = torch.tensor([[0.0]], dtype=torch.float32, device=DEVICE)
        if random.random() < eps:
            action = random.randrange(RouletteEnv.N_ACTIONS)
        else:
            with torch.no_grad():
                action = int(policy(state).argmax(dim=1).item())

        _, reward = env.step(action)
        rewards.append(reward)
        replay.append((0.0, action, reward))

        if len(replay) >= batch_size:
            batch = random.sample(replay, batch_size)
            states = torch.zeros((batch_size, 1), dtype=torch.float32, device=DEVICE)
            actions = torch.tensor([x[1] for x in batch], dtype=torch.long, device=DEVICE)
            targets = torch.tensor([x[2] for x in batch], dtype=torch.float32, device=DEVICE)

            predicted = policy(states).gather(1, actions.unsqueeze(1)).squeeze(1)
            with torch.no_grad():
                # gamma=0: the target is the immediate reward.
                expected = targets + gamma * target(states).max(dim=1).values
            loss = nn.functional.smooth_l1_loss(predicted, expected)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        if (step + 1) % target_sync == 0:
            target.load_state_dict(policy.state_dict())

    with torch.no_grad():
        learned_values = policy(torch.tensor([[0.0]], dtype=torch.float32, device=DEVICE)).cpu().numpy()[0]
    return policy, learned_values, np.asarray(rewards)

roulette_dqn, roulette_dqn_q, roulette_drl_rewards = train_roulette_dqn()
print("DQN estimated action values:", np.round(roulette_dqn_q, 3))
print("Best estimated action:", int(np.argmax(roulette_dqn_q)))
def moving_average(values, window=200):
    values = np.asarray(values, dtype=float)
    if len(values) < window:
        window = max(1, len(values))
    return np.convolve(values, np.ones(window) / window, mode="valid")

plt.figure(figsize=(10, 4))
plt.plot(moving_average(roulette_rl_rewards), label="Q-Learning")
plt.plot(moving_average(roulette_drl_rewards), label="DQN")
plt.title("Roulette: moving average reward")
plt.xlabel("Episode / spin")
plt.ylabel("Average reward")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

print("Mean reward -- Q-Learning:", round(float(roulette_rl_rewards.mean()), 4))
print("Mean reward -- DQN:", round(float(roulette_drl_rewards.mean()), 4))

In [ ]:
def make_mountaincar(render_mode=None):
    return gym.make("MountainCar-v0", render_mode=render_mode)

env_check = make_mountaincar()
print("Observation space:", env_check.observation_space)
print("Action space:", env_check.action_space)
env_check.close()
def discretize_mountaincar_state(state, bins):
    return tuple(
        int(np.clip(np.digitize(state[i], bins[i]), 0, len(bins[i])))
        for i in range(len(state))
    )


def train_mountaincar_qlearning(episodes=5000, alpha=0.1, gamma=0.99,
                                epsilon=1.0, epsilon_min=0.02,
                                epsilon_decay=0.995):
    env = make_mountaincar()
    bins = [
        np.linspace(-1.2, 0.6, 20),
        np.linspace(-0.07, 0.07, 20)
    ]
    q_table = np.zeros((21, 21, env.action_space.n), dtype=np.float32)
    episode_rewards, episode_steps = [], []

    for ep in range(episodes):
        state, _ = env.reset(seed=SEED + ep)
        s = discretize_mountaincar_state(state, bins)
        total_reward = 0

        for step in range(200):
            if random.random() < epsilon:
                action = env.action_space.sample()
            else:
                action = int(np.argmax(q_table[s]))

            next_state, reward, terminated, truncated, _ = env.step(action)
            ns = discretize_mountaincar_state(next_state, bins)
            done = terminated or truncated
            best_next = 0.0 if done else np.max(q_table[ns])
            q_table[s + (action,)] += alpha * (
                reward + gamma * best_next - q_table[s + (action,)]
            )
            s = ns
            total_reward += reward
            if done:
                break

        episode_rewards.append(total_reward)
        episode_steps.append(step + 1)
        epsilon = max(epsilon_min, epsilon * epsilon_decay)

    env.close()
    return q_table, bins, np.asarray(episode_rewards), np.asarray(episode_steps)

mc_q_table, mc_bins, mc_rl_rewards, mc_rl_steps = train_mountaincar_qlearning()
print("Q-table shape:", mc_q_table.shape)
print("Last-100 episode mean reward:", round(float(mc_rl_rewards[-100:].mean()), 2))
Transition = namedtuple("Transition", ["state", "action", "reward", "next_state", "done"])

class ReplayMemory:
    def __init__(self, capacity=50000):
        self.memory = deque(maxlen=capacity)
    def push(self, *args):
        self.memory.append(Transition(*args))
    def sample(self, batch_size):
        return random.sample(self.memory, batch_size)
    def __len__(self):
        return len(self.memory)


class MountainCarDQN(nn.Module):
    def __init__(self, n_states=2, n_actions=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_states, 128), nn.ReLU(),
            nn.Linear(128, 128), nn.ReLU(),
            nn.Linear(128, n_actions)
        )
    def forward(self, x):
        return self.net(x)


def train_mountaincar_dqn(episodes=700, batch_size=128, gamma=0.99,
                          lr=1e-3, epsilon_start=1.0, epsilon_end=0.02,
                          epsilon_decay=8000, target_sync=10):
    env = make_mountaincar()
    policy = MountainCarDQN().to(DEVICE)
    target = MountainCarDQN().to(DEVICE)
    target.load_state_dict(policy.state_dict())
    optimizer = optim.AdamW(policy.parameters(), lr=lr)
    memory = ReplayMemory()
    rewards, steps_per_episode = [], []
    global_steps = 0

    for ep in range(episodes):
        state, _ = env.reset(seed=SEED + ep)
        total_reward = 0

        for step in range(200):
            eps = epsilon_end + (epsilon_start - epsilon_end) * np.exp(-global_steps / epsilon_decay)
            global_steps += 1
            state_tensor = torch.tensor(state, dtype=torch.float32, device=DEVICE).unsqueeze(0)

            if random.random() < eps:
                action = env.action_space.sample()
            else:
                with torch.no_grad():
                    action = int(policy(state_tensor).argmax(dim=1).item())

            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            memory.push(state.copy(), action, reward, next_state.copy(), done)
            state = next_state
            total_reward += reward

            if len(memory) >= batch_size:
                batch = memory.sample(batch_size)
                states = torch.tensor(np.array([t.state for t in batch]), dtype=torch.float32, device=DEVICE)
                actions = torch.tensor([t.action for t in batch], dtype=torch.long, device=DEVICE).unsqueeze(1)
                batch_rewards = torch.tensor([t.reward for t in batch], dtype=torch.float32, device=DEVICE)
                next_states = torch.tensor(np.array([t.next_state for t in batch]), dtype=torch.float32, device=DEVICE)
                dones = torch.tensor([t.done for t in batch], dtype=torch.float32, device=DEVICE)

                current_q = policy(states).gather(1, actions).squeeze(1)
                with torch.no_grad():
                    next_q = target(next_states).max(dim=1).values
                    expected_q = batch_rewards + gamma * next_q * (1 - dones)

                loss = nn.functional.smooth_l1_loss(current_q, expected_q)
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(policy.parameters(), 10)
                optimizer.step()

            if done:
                break

        rewards.append(total_reward)
        steps_per_episode.append(step + 1)
        if (ep + 1) % target_sync == 0:
            target.load_state_dict(policy.state_dict())

    env.close()
    return policy, np.asarray(rewards), np.asarray(steps_per_episode)

mc_dqn, mc_drl_rewards, mc_drl_steps = train_mountaincar_dqn()
print("Last-100 episode mean reward:", round(float(mc_drl_rewards[-100:].mean()), 2))
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
axes[0, 0].plot(mc_rl_rewards)
axes[0, 0].set_title("Q-Learning: reward per episode")
axes[0, 1].plot(mc_rl_steps)
axes[0, 1].set_title("Q-Learning: steps per episode")
axes[1, 0].plot(mc_drl_rewards)
axes[1, 0].set_title("DQN: reward per episode")
axes[1, 1].plot(mc_drl_steps)
axes[1, 1].set_title("DQN: steps per episode")
for ax in axes.flat:
    ax.set_xlabel("Episode")
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()
def evaluate_mountaincar_qlearning(q_table, bins, episodes=5):
    env = make_mountaincar()
    results = []
    for _ in range(episodes):
        state, _ = env.reset()
        s = discretize_mountaincar_state(state, bins)
        total = 0
        for step in range(200):
            action = int(np.argmax(q_table[s]))
            state, reward, terminated, truncated, _ = env.step(action)
            s = discretize_mountaincar_state(state, bins)
            total += reward
            if terminated or truncated:
                break
        results.append((total, step + 1, bool(terminated)))
    env.close()
    return results


def evaluate_mountaincar_dqn(policy, episodes=5):
    env = make_mountaincar()
    results = []
    policy.eval()
    for _ in range(episodes):
        state, _ = env.reset()
        total = 0
        for step in range(200):
            x = torch.tensor(state, dtype=torch.float32, device=DEVICE).unsqueeze(0)
            with torch.no_grad():
                action = int(policy(x).argmax(dim=1).item())
            state, reward, terminated, truncated, _ = env.step(action)
            total += reward
            if terminated or truncated:
                break
        results.append((total, step + 1, bool(terminated)))
    env.close()
    return results

print("Q-Learning evaluation (reward, steps, reached_goal):")
print(evaluate_mountaincar_qlearning(mc_q_table, mc_bins))
print("\nDQN evaluation (reward, steps, reached_goal):")
print(evaluate_mountaincar_dqn(mc_dqn))

In [ ]:
def build_user_graph():
    edge_text = input("Enter edges (example: A-B, A-C, B-D, C-D, D-E): ").strip()
    edges = []
    for item in edge_text.split(","):
        item = item.strip()
        if not item:
            continue
        parts = [p.strip() for p in item.split("-")]
        if len(parts) != 2 or not all(parts):
            raise ValueError(f"Invalid edge: {item!r}. Use the format node-node.")
        edges.append((parts[0], parts[1]))

    graph = nx.Graph()
    graph.add_edges_from(edges)
    if graph.number_of_nodes() == 0:
        raise ValueError("The graph must contain at least one edge.")

    start = input(f"Start node {list(graph.nodes())}: ").strip()
    goal = input(f"Goal node {list(graph.nodes())}: ").strip()
    if start not in graph or goal not in graph:
        raise ValueError("Start and goal must both be present in the graph.")
    if not nx.has_path(graph, start, goal):
        raise ValueError("There is no path between the selected start and goal.")
    return graph, start, goal

# Run this cell and enter your own graph when prompted.
G, START, GOAL = build_user_graph()
print("Nodes:", list(G.nodes()))
print("Edges:", list(G.edges()))
print("Start:", START, "| Goal:", GOAL)
class GraphEnvironment:
    def __init__(self, graph, start, goal):
        self.graph = graph
        self.nodes = list(graph.nodes())
        self.node_to_idx = {node: i for i, node in enumerate(self.nodes)}
        self.idx_to_node = {i: node for node, i in self.node_to_idx.items()}
        self.start_idx = self.node_to_idx[start]
        self.goal_idx = self.node_to_idx[goal]
        self.n = len(self.nodes)

    def reset(self):
        self.state = self.start_idx
        return self.state

    def valid_actions(self, state):
        node = self.idx_to_node[state]
        return [self.node_to_idx[n] for n in self.graph.neighbors(node)]

    def step(self, action):
        if action not in self.valid_actions(self.state):
            return self.state, -10.0, False
        self.state = action
        if self.state == self.goal_idx:
            return self.state, 100.0, True
        return self.state, -1.0, False

    def shortest_path(self):
        return nx.shortest_path(
            self.graph,
            source=self.idx_to_node[self.start_idx],
            target=self.idx_to_node[self.goal_idx]
        )


def extract_greedy_path(env, q_function, is_dqn=False, max_steps=None):
    state = env.reset()
    path = [env.idx_to_node[state]]
    visited = {state}
    max_steps = max_steps or env.n * 2

    for _ in range(max_steps):
        if state == env.goal_idx:
            return path
        valid = env.valid_actions(state)
        if is_dqn:
            one_hot = torch.zeros((1, env.n), dtype=torch.float32, device=DEVICE)
            one_hot[0, state] = 1.0
            with torch.no_grad():
                values = q_function(one_hot).squeeze(0).cpu().numpy()
        else:
            values = q_function[state]

        action = max(valid, key=lambda a: values[a])
        if action in visited and action != env.goal_idx:
            break
        state, _, done = env.step(action)
        path.append(env.idx_to_node[state])
        visited.add(state)
        if done:
            break
    return path
def train_graph_qlearning(env, episodes=3000, alpha=0.1, gamma=0.95,
                          epsilon=1.0, epsilon_min=0.02, epsilon_decay=0.995,
                          max_steps=100):
    q_table = np.zeros((env.n, env.n), dtype=np.float32)
    start_time = time.perf_counter()

    for _ in range(episodes):
        state = env.reset()
        for _ in range(max_steps):
            valid = env.valid_actions(state)
            if random.random() < epsilon:
                action = random.choice(valid)
            else:
                action = max(valid, key=lambda a: q_table[state, a])

            next_state, reward, done = env.step(action)
            future = 0.0 if done else max(q_table[next_state, a] for a in env.valid_actions(next_state))
            q_table[state, action] += alpha * (
                reward + gamma * future - q_table[state, action]
            )
            state = next_state
            if done:
                break
        epsilon = max(epsilon_min, epsilon * epsilon_decay)

    return q_table, time.perf_counter() - start_time
class GraphDQN(nn.Module):
    def __init__(self, n_nodes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_nodes, 64), nn.ReLU(),
            nn.Linear(64, 64), nn.ReLU(),
            nn.Linear(64, n_nodes)
        )
    def forward(self, x):
        return self.net(x)


def one_hot_state(state, n, device=DEVICE):
    x = torch.zeros((1, n), dtype=torch.float32, device=device)
    x[0, state] = 1.0
    return x


def train_graph_dqn(env, episodes=3000, gamma=0.95, lr=1e-3,
                    epsilon_start=1.0, epsilon_min=0.02,
                    epsilon_decay=0.995, batch_size=64,
                    target_sync=20, max_steps=100):
    policy = GraphDQN(env.n).to(DEVICE)
    target = GraphDQN(env.n).to(DEVICE)
    target.load_state_dict(policy.state_dict())
    optimizer = optim.AdamW(policy.parameters(), lr=lr)
    replay = deque(maxlen=10000)
    epsilon = epsilon_start
    start_time = time.perf_counter()

    for episode in range(episodes):
        state = env.reset()
        for _ in range(max_steps):
            valid = env.valid_actions(state)
            if random.random() < epsilon:
                action = random.choice(valid)
            else:
                with torch.no_grad():
                    q = policy(one_hot_state(state, env.n)).squeeze(0)
                    # Mask invalid moves so exploitation always selects a neighbor.
                    mask = torch.full_like(q, -1e9)
                    mask[valid] = 0
                    action = int((q + mask).argmax().item())

            next_state, reward, done = env.step(action)
            replay.append((state, action, reward, next_state, done))
            state = next_state

            if len(replay) >= batch_size:
                batch = random.sample(replay, batch_size)
                states = torch.zeros((batch_size, env.n), device=DEVICE)
                next_states = torch.zeros((batch_size, env.n), device=DEVICE)
                actions = torch.tensor([b[1] for b in batch], dtype=torch.long, device=DEVICE)
                rewards = torch.tensor([b[2] for b in batch], dtype=torch.float32, device=DEVICE)
                dones = torch.tensor([b[4] for b in batch], dtype=torch.float32, device=DEVICE)
                for i, b in enumerate(batch):
                    states[i, b[0]] = 1.0
                    next_states[i, b[3]] = 1.0

                current = policy(states).gather(1, actions.unsqueeze(1)).squeeze(1)
                with torch.no_grad():
                    next_values = torch.zeros(batch_size, device=DEVICE)
                    for i, b in enumerate(batch):
                        if not b[4]:
                            next_valid = env.valid_actions(b[3])
                            next_q = target(next_states[i:i+1]).squeeze(0)
                            next_values[i] = max(next_q[a].item() for a in next_valid)
                    expected = rewards + gamma * next_values * (1 - dones)

                loss = nn.functional.smooth_l1_loss(current, expected)
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(policy.parameters(), 5.0)
                optimizer.step()

            if done:
                break

        epsilon = max(epsilon_min, epsilon * epsilon_decay)
        if (episode + 1) % target_sync == 0:
            target.load_state_dict(policy.state_dict())

    return policy, time.perf_counter() - start_time
# Re-create independent environments for fair, repeatable training.
env_rl = GraphEnvironment(G, START, GOAL)
env_drl = GraphEnvironment(G, START, GOAL)

optimal_path = env_rl.shortest_path()
q_table_graph, rl_train_seconds = train_graph_qlearning(env_rl)
dqn_graph, drl_train_seconds = train_graph_dqn(env_drl)

rl_path = extract_greedy_path(env_rl, q_table_graph, is_dqn=False)
drl_path = extract_greedy_path(env_drl, dqn_graph, is_dqn=True)

def path_length(path, goal):
    return len(path) - 1 if path and path[-1] == goal else None

rl_length = path_length(rl_path, GOAL)
drl_length = path_length(drl_path, GOAL)
optimal_length = len(optimal_path) - 1

print("Dijkstra optimal path:", " -> ".join(optimal_path), "| length:", optimal_length)
print("RL Q-Learning path:", " -> ".join(rl_path), "| length:", rl_length)
print("DRL DQN path:", " -> ".join(drl_path), "| length:", drl_length)
def parameter_count(model):
    return sum(p.numel() for p in model.parameters())

comparison = [
    ["Path", " -> ".join(rl_path), " -> ".join(drl_path)],
    ["Path length (edges)", rl_length if rl_length is not None else "Not reached",
     drl_length if drl_length is not None else "Not reached"],
    ["Optimal length (Dijkstra)", optimal_length, optimal_length],
    ["Is shortest path?", rl_length == optimal_length, drl_length == optimal_length],
    ["Training time (seconds)", round(rl_train_seconds, 4), round(drl_train_seconds, 4)],
    ["Learned representation", f"Q-table: {q_table_graph.shape}",
     f"Neural network: {parameter_count(dqn_graph)} parameters"],
    ["Learning method", "Bellman table update", "Backpropagation + replay buffer"],
]
print(f"{'Metric':<27} | {'RL (Q-Learning)':<35} | {'DRL (DQN)':<35}")
print("-" * 105)
for row in comparison:
    print(f"{str(row[0]):<27} | {str(row[1]):<35} | {str(row[2]):<35}")

Enter edges (example: A-B, A-C, B-D, C-D, D-E): A-B, B-D, A-C, C-E
Start node ['A', 'B', 'D', 'C', 'E']: A
Goal node ['A', 'B', 'D', 'C', 'E']: E
Nodes: ['A', 'B', 'D', 'C', 'E']
Edges: [('A', 'B'), ('A', 'C'), ('B', 'D'), ('C', 'E')]
Start: A | Goal: E
Dijkstra optimal path: A -> C -> E | length: 2
RL Q-Learning path: A -> C -> E | length: 2
DRL DQN path: A -> C -> E | length: 2
Metric                      | RL (Q-Learning)                     | DRL (DQN)                          
---------------------------------------------------------------------------------------------------------
Path                        | A -> C -> E                         | A -> C -> E                        
Path length (edges)         | 2                                   | 2                                  
Optimal length (Dijkstra)   | 2                                   | 2                                  
Is shortest path?           | True                                | True                       